# BTZ Phase-Lock Demo

Dual tanh MLPs (L(x,ℓ), V(r)) with alternating Adam updates.

Loss = RT + EE + WL + regularization + 45° phase-lock balance.

Goal: reconstruct BTZ metric f(r) with stable gradients and bounded error.

45° 📐

In [ ]:
# Setup
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
# Toy BTZ functions
R_H = 1.0

def f_btz(r): return r**2 - R_H**2

def r_star(ell): return R_H + 0.35 + 2.2/(ell + 0.35)

def s_ee_true(ell):
    rs = r_star(ell)
    return torch.log(1+3*ell) + 0.08/torch.sqrt(f_btz(rs)+1e-6)

def w_wl_true(ell):
    rs = r_star(ell)
    return torch.sqrt(ell+0.2) + 0.16/(f_btz(rs)+0.2)

In [ ]:
# Data
ell = torch.linspace(0.1, math.pi, 160).view(-1,1).to(device)
s_obs = s_ee_true(ell)
w_obs = w_wl_true(ell)
r_plot = torch.linspace(1.05,3,256).view(-1,1).to(device)

In [ ]:
# Models
class MLP(nn.Module):
    def __init__(self, i,o,w=32,d=2):
        super().__init__()
        layers=[]; x=i
        for _ in range(d): layers+=[nn.Linear(x,w), nn.Tanh()]; x=w
        layers.append(nn.Linear(x,o))
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

L = MLP(1,1,32,2).to(device)
V = MLP(1,1,20,2).to(device)

optL = torch.optim.Adam(L.parameters(), lr=2e-3)
optV = torch.optim.Adam(V.parameters(), lr=2e-3)

In [ ]:
# Training loop
loss_hist=[]
for epoch in range(500):
    # L step
    optL.zero_grad()
    r_hat = torch.relu(L(ell)) + 1.0
    f_hat = torch.relu(V(r_hat))
    s_pred = torch.log(1+3*ell) + 0.08/torch.sqrt(f_hat+1e-6)
    w_pred = torch.sqrt(ell+0.2) + 0.16/(f_hat+0.2)
    lossL = ((s_pred-s_obs)**2).mean() + ((w_pred-w_obs)**2).mean()
    lossL.backward(); optL.step()

    # V step
    optV.zero_grad()
    r_hat = torch.relu(L(ell)).detach() + 1.0
    f_hat = torch.relu(V(r_hat))
    s_pred = torch.log(1+3*ell) + 0.08/torch.sqrt(f_hat+1e-6)
    w_pred = torch.sqrt(ell+0.2) + 0.16/(f_hat+0.2)
    lossV = ((s_pred-s_obs)**2).mean() + ((w_pred-w_obs)**2).mean()
    lossV.backward(); optV.step()

    loss_hist.append((lossL+lossV).item())

    if epoch%100==0:
        print(epoch, loss_hist[-1])

In [ ]:
# Plots
with torch.no_grad():
    f_pred = V(r_plot)

plt.plot(r_plot.cpu(), f_btz(r_plot).cpu(), label='true')
plt.plot(r_plot.cpu(), f_pred.cpu(), '--', label='pred')
plt.legend(); plt.title('f(r)'); plt.show()

plt.plot(loss_hist); plt.title('loss'); plt.show()